# 09. Blancos probables de hostilidad y odio

Este notebook estima hacia qué personas, partidos, instituciones, grupos ideológicos o medios podrían orientarse los comentarios hostiles del corpus formal de HateCR.

La salida es una **inferencia exploratoria para validación manual**, no una atribución definitiva del destinatario ni una medición automática de discurso de odio.

## 1. Objetivo y niveles de evidencia

Se distinguen cinco resultados mutuamente excluyentes por comentario:

1. **Alta confianza:** una ofensa validada aparece vinculada localmente a una única entidad mencionada.
2. **Confianza media:** existe una única mención directa, pero no un vínculo lingüístico suficiente con la ofensa.
3. **Confianza baja:** el comentario no nombra el blanco y el post madre menciona una única entidad catalogada.
4. **Ambiguo:** aparecen varios blancos posibles.
5. **Sin asignar:** no existe evidencia suficiente.

Los handles de medios se conservan para auditoría, pero no generan automáticamente un blanco: suelen formar parte del formato normal de una reply.

## 2. Entradas y salidas

**Entradas**

- `data/processed/x_media_anchored_interactions_corpus_formal_predictions_v2.csv`
- `data/interim/source_posts_formal_unique.csv`
- `config/target_entities.yaml`
- `config/media_accounts.yaml`
- `config/political_accounts.yaml`
- `reports/formal_lexical/hostility_v2_common_direct_offenses.csv`

**Salidas**

- Tablas y muestra de revisión en `reports/formal_targets/`
- Figuras en `reports/formal_targets/figures/`
- Corpus enriquecido en `data/processed/x_media_anchored_interactions_corpus_formal_targets_exploratory.csv`

No se llama a la API de X y no se modifica `reports/formal_eda/manual_review_sample.csv`.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, display


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raíz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.target_analysis import (
    build_assignment_coverage,
    build_target_assignments,
    build_offense_target_pairs,
    build_primary_target_summary,
    build_relation_templates,
    build_target_manual_review_candidate,
    build_target_scope_summary,
    build_target_type_summary,
    count_catalog_mentions,
    discover_uncatalogued_source_entities,
    enrich_dependency_evidence,
    extract_target_evidence,
    load_target_catalog,
    load_validated_offense_patterns,
    prepare_target_inputs,
    save_anchor_context_rate_figure,
    save_assignment_coverage_figure,
    save_direct_target_mentions_figure,
    save_entity_mention_hostility_figure,
    try_load_spanish_spacy_model,
)

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIG_DIR = PROJECT_ROOT / "config"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_targets"
FIGURES_DIR = REPORTS_DIR / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

## 3. Parámetros reproducibles

El umbral de ocho tokens es una regla inicial. spaCy comprueba además si la ofensa y la entidad pertenecen a la misma oración y calcula su distancia en el árbol de dependencias.

In [ ]:
CORPUS_PATH = Path(
    os.getenv(
        "TARGET_CORPUS_PATH",
        str(DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_predictions_v2.csv"),
    )
)
SOURCE_POSTS_PATH = DATA_INTERIM / "source_posts_formal_unique.csv"
TARGET_CONFIG_PATH = CONFIG_DIR / "target_entities.yaml"
MEDIA_CONFIG_PATH = CONFIG_DIR / "media_accounts.yaml"
POLITICAL_CONFIG_PATH = CONFIG_DIR / "political_accounts.yaml"
OFFENSE_PATTERNS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "formal_lexical"
    / "hostility_v2_common_direct_offenses.csv"
)
HOSTILITY_COLUMN = os.getenv(
    "TARGET_HOSTILITY_COLUMN", "ml_hostility_pred_v2_balanced"
)
HATE_COLUMN = os.getenv(
    "TARGET_HATE_COLUMN", "ml_hate_speech_pred_v2_balanced_experimental"
)
MAX_OFFENSE_TARGET_TOKEN_DISTANCE = int(
    os.getenv("MAX_OFFENSE_TARGET_TOKEN_DISTANCE", "8")
)
MAX_DEPENDENCY_DISTANCE = int(os.getenv("MAX_DEPENDENCY_DISTANCE", "4"))
MIN_CONTEXT_COMMENTS = int(os.getenv("MIN_TARGET_CONTEXT_COMMENTS", "30"))
TARGET_REVIEW_SAMPLE_SIZE = int(os.getenv("TARGET_REVIEW_SAMPLE_SIZE", "300"))
NER_MIN_SOURCE_POSTS = int(os.getenv("NER_MIN_SOURCE_POSTS", "3"))
ENABLE_SPACY = os.getenv("ENABLE_TARGET_SPACY", "true").lower() == "true"
ENABLE_NER_DISCOVERY = os.getenv("ENABLE_TARGET_NER_DISCOVERY", "true").lower() == "true"
RANDOM_STATE = 42

parameters_df = pd.DataFrame(
    [
        {"parameter": "corpus", "value": str(CORPUS_PATH)},
        {"parameter": "hostility_column", "value": HOSTILITY_COLUMN},
        {"parameter": "hate_column", "value": HATE_COLUMN},
        {"parameter": "max_token_distance", "value": MAX_OFFENSE_TARGET_TOKEN_DISTANCE},
        {"parameter": "max_dependency_distance", "value": MAX_DEPENDENCY_DISTANCE},
        {"parameter": "min_context_comments", "value": MIN_CONTEXT_COMMENTS},
        {"parameter": "review_sample_size", "value": TARGET_REVIEW_SAMPLE_SIZE},
        {"parameter": "enable_spacy", "value": ENABLE_SPACY},
        {"parameter": "enable_ner_discovery", "value": ENABLE_NER_DISCOVERY},
    ]
)
display(parameters_df)

## 4. Carga segura y validaciones

In [ ]:
def safe_read_csv(path, name, dtype=None):
    path = Path(path)
    if not path.exists():
        print("[ADVERTENCIA] No existe {}: {}".format(name, path))
        return pd.DataFrame()
    frame = pd.read_csv(path, dtype=dtype)
    print("[OK] {}: {:,} filas".format(name, len(frame)))
    return frame


corpus_df = safe_read_csv(
    CORPUS_PATH,
    "corpus formal v2",
    dtype={"tweet_id": "string", "anchor_post_id": "string"},
)
source_posts_df = safe_read_csv(
    SOURCE_POSTS_PATH,
    "posts madre formales",
    dtype={"source_post_id": "string"},
)
if corpus_df.empty or source_posts_df.empty:
    raise FileNotFoundError("Faltan entradas obligatorias para el análisis de blancos")

required_corpus_columns = {
    "tweet_id",
    "anchor_post_id",
    "text",
    HOSTILITY_COLUMN,
    HATE_COLUMN,
}
missing = sorted(required_corpus_columns.difference(corpus_df.columns))
if missing:
    raise KeyError("Faltan columnas del corpus: {}".format(missing))

validation_df = pd.DataFrame(
    [
        {"check": "corpus_rows", "value": len(corpus_df)},
        {"check": "unique_tweets", "value": corpus_df["tweet_id"].nunique()},
        {"check": "duplicate_tweet_ids", "value": int(corpus_df["tweet_id"].duplicated().sum())},
        {"check": "source_posts", "value": source_posts_df["source_post_id"].nunique()},
        {
            "check": "comments_with_source_post",
            "value": int(corpus_df["anchor_post_id"].isin(source_posts_df["source_post_id"]).sum()),
        },
        {
            "check": "hostility_balanced_positive",
            "value": int(pd.to_numeric(corpus_df[HOSTILITY_COLUMN], errors="coerce").fillna(0).sum()),
        },
        {
            "check": "hate_balanced_experimental_positive",
            "value": int(pd.to_numeric(corpus_df[HATE_COLUMN], errors="coerce").fillna(0).sum()),
        },
    ]
)
display(validation_df)

## 5. Catálogo de blancos y ofensas validadas

El catálogo agrupa variantes ortográficas bajo una entidad canónica. Los alias de contexto, como ciertos apellidos, solo se aplican al texto periodístico del post madre y nunca elevan por sí mismos la confianza.

In [ ]:
target_catalog_df = load_target_catalog(
    TARGET_CONFIG_PATH,
    media_accounts_path=MEDIA_CONFIG_PATH,
    political_accounts_path=POLITICAL_CONFIG_PATH,
)
offense_patterns_df = load_validated_offense_patterns(OFFENSE_PATTERNS_PATH)

catalog_summary_df = (
    target_catalog_df.groupby("target_type", as_index=False)
    .agg(n_targets=("target_id", "nunique"))
    .sort_values("n_targets", ascending=False)
)
display(catalog_summary_df)
print("Ofensas directas validadas:", len(offense_patterns_df))

## 6. Extracción de evidencia y análisis lingüístico

Primero se detectan aliases y ofensas mediante expresiones regulares normalizadas. Después, si spaCy está disponible, se verifica oración compartida y distancia sintáctica solamente en los comentarios candidatos.

In [ ]:
prepared_df = prepare_target_inputs(
    corpus_df,
    source_posts_df,
    hostility_prediction_column=HOSTILITY_COLUMN,
    hate_prediction_column=HATE_COLUMN,
)
evidence_df = extract_target_evidence(
    prepared_df,
    target_catalog_df,
    offense_patterns_df,
    max_token_distance=MAX_OFFENSE_TARGET_TOKEN_DISTANCE,
)

nlp = try_load_spanish_spacy_model() if ENABLE_SPACY else None
if nlp is None:
    print("[ADVERTENCIA] No hay modelo español de spaCy; se usará proximidad léxica.")
else:
    print("[OK] spaCy:", nlp.meta.get("name", "modelo español"))
    evidence_df = enrich_dependency_evidence(
        evidence_df,
        prepared_df,
        nlp,
        max_dependency_distance=MAX_DEPENDENCY_DISTANCE,
    )

assignments_df = build_target_assignments(
    prepared_df,
    evidence_df,
    target_catalog_df,
)

print("Filas de evidencia:", len(evidence_df))
print("Comentarios con alguna evidencia:", evidence_df["tweet_id"].nunique())
print("Asignaciones finales:", len(assignments_df))

## 6A. Censo de menciones políticas en todo el corpus

Este conteo no filtra por hostilidad, odio, evento ni medio. Incluye personas, partidos, instituciones y grupos ideológicos; excluye medios para evitar contar handles automáticos de replies. Una mención sigue sin equivaler a un ataque.

Para esta tabla, `Fraude Amplio` se registra como alias de conteo de `Frente Amplio`. Este alias no interviene en la clasificación de hostilidad u odio ni eleva la confianza del blanco inferido.


In [ ]:
mention_target_summary_df, mention_type_summary_df, mention_overall_summary_df = count_catalog_mentions(
    prepared_df,
    target_catalog_df,
    target_types=("person", "party", "institution", "ideological_group"),
)

print("Total general de menciones políticas:")
display(mention_overall_summary_df.round(2))
print("Totales por tipo:")
display(mention_type_summary_df.round(2))
print("Entidades más mencionadas:")
display(mention_target_summary_df.head(30).round(2))


## 6B. Entidades vinculadas con comentarios hostiles

El porcentaje usa como denominador los **comentarios únicos que mencionan cada entidad**. Una entidad repetida varias veces dentro del mismo comentario cuenta una sola vez. El gráfico presenta los conteos y porcentajes exactos, sin intervalos de confianza.

> **Límite metodológico:** esta es una coaparición entre mención y predicción de hostilidad del perfil balanceado v2. No demuestra que la entidad sea el destinatario del ataque; esa relación requiere análisis contextual o validación manual.


In [ ]:
SELECTED_HOSTILITY_ENTITY_IDS = [
    "pln",
    "tse",
    "laura_fernandez",
    "izquierda_comunismo",
    "chavismo",
    "ppso",
    "frente_amplio",
    "pusc",
    "populismo",
    "oficialismo",
    "rodrigo_chaves",
]
ENTITY_DISPLAY_LABELS = {
    "pln": "PLN",
    "tse": "TSE",
    "laura_fernandez": "Laura Fernández",
    "izquierda_comunismo": "Izquierda / comunismo",
    "chavismo": "Chavismo / personas chavistas",
    "ppso": "Pueblo Soberano / PPSO",
    "frente_amplio": "Frente Amplio",
    "pusc": "PUSC",
    "populismo": "Populismo / personas populistas",
    "oficialismo": "Oficialismo",
    "rodrigo_chaves": "Rodrigo Chaves",
}
entity_order = {
    target_id: index for index, target_id in enumerate(SELECTED_HOSTILITY_ENTITY_IDS)
}
entity_mention_hostility_df = mention_target_summary_df.loc[
    mention_target_summary_df["target_id"].isin(SELECTED_HOSTILITY_ENTITY_IDS)
].copy()
entity_mention_hostility_df["entity"] = entity_mention_hostility_df["target_id"].map(
    ENTITY_DISPLAY_LABELS
)
entity_mention_hostility_df["display_order"] = entity_mention_hostility_df["target_id"].map(
    entity_order
)
entity_mention_hostility_df = entity_mention_hostility_df.sort_values(
    "display_order"
).reset_index(drop=True)
entity_mention_hostility_df = entity_mention_hostility_df[
    [
        "target_id",
        "entity",
        "target_type",
        "mention_occurrences",
        "comments_mentioning_entity",
        "hostile_comments_mentioning_entity",
        "hostility_pct_of_entity_comments",
        "hostility_ci95_low",
        "hostility_ci95_high",
    ]
]

entity_table_display = entity_mention_hostility_df.rename(
    columns={
        "entity": "Entidad",
        "mention_occurrences": "Menciones",
        "comments_mentioning_entity": "Comentarios únicos con mención",
        "hostile_comments_mentioning_entity": "Comentarios hostiles",
        "hostility_pct_of_entity_comments": "% vinculado con hostilidad",
    }
)[
    [
        "Entidad",
        "Menciones",
        "Comentarios únicos con mención",
        "Comentarios hostiles",
        "% vinculado con hostilidad",
    ]
]
display(entity_table_display.round(2))

entity_mention_plot_df = mention_target_summary_df.copy()
entity_mention_plot_df["target_label"] = entity_mention_plot_df["target_id"].map(
    ENTITY_DISPLAY_LABELS
).fillna(entity_mention_plot_df["target_label"])
ENTITY_HOSTILITY_FIGURE = save_entity_mention_hostility_figure(
    entity_mention_plot_df,
    SELECTED_HOSTILITY_ENTITY_IDS,
    FIGURES_DIR / "entity_mention_hostility_percentage.png",
)
display(Image(filename=str(ENTITY_HOSTILITY_FIGURE)))


## 7. Cobertura y confianza

La confianza baja es útil para orientar revisión, pero no debe presentarse como destinatario confirmado. Los casos ambiguos conservan todos los candidatos sin escoger arbitrariamente uno.

In [ ]:
coverage_df = build_assignment_coverage(assignments_df)
display(coverage_df.round(2))

COVERAGE_FIGURE = save_assignment_coverage_figure(
    coverage_df,
    FIGURES_DIR / "target_assignment_confidence_coverage.png",
)
display(Image(filename=str(COVERAGE_FIGURE)))

## 8. Resultados por entidad y tipo de blanco

Se generan tres vistas distintas:

- **Mención directa:** la entidad aparece en el comentario.
- **Vínculo directo:** una ofensa validada y la entidad comparten una relación local plausible.
- **Contexto del post madre:** la entidad aparece en la publicación periodística que origina la conversación.

Estas vistas no deben mezclarse porque expresan niveles de evidencia diferentes.

In [ ]:
scope_summary_df = build_target_scope_summary(evidence_df)
primary_target_summary_df = build_primary_target_summary(assignments_df)
target_type_summary_df = build_target_type_summary(assignments_df)

print("Resumen por tipo de blanco asignado:")
display(target_type_summary_df.round(2))

print("Principales menciones directas dentro de comentarios hostiles:")
direct_hostile_df = (
    scope_summary_df.loc[scope_summary_df["evidence_scope"].eq("direct_mention")]
    .sort_values("hostile_n", ascending=False)
    .head(25)
)
display(
    direct_hostile_df[
        [
            "target_label",
            "target_type",
            "n_comments",
            "hostile_n",
            "hostility_pct",
            "hate_n_experimental",
        ]
    ].round(2)
)

DIRECT_FIGURE = save_direct_target_mentions_figure(
    scope_summary_df,
    FIGURES_DIR / "direct_targets_in_hostile_comments.png",
    top_n=15,
)
display(Image(filename=str(DIRECT_FIGURE)))

## 9. Contexto del post madre y normalización por exposición

Un volumen alto puede significar simplemente que un actor recibió más cobertura periodística. Por ello se muestra también la proporción hostil entre todos los comentarios asociados a posts madre que mencionan cada entidad.

In [ ]:
context_df = scope_summary_df.loc[
    scope_summary_df["evidence_scope"].eq("anchor_context")
    & scope_summary_df["n_comments"].ge(MIN_CONTEXT_COMMENTS)
].copy()
display(
    context_df.sort_values("hostility_pct", ascending=False)[
        [
            "target_label",
            "target_type",
            "n_comments",
            "hostile_n",
            "hostility_pct",
            "hate_n_experimental",
            "hate_pct_experimental",
        ]
    ].head(25).round(2)
)

CONTEXT_FIGURE = save_anchor_context_rate_figure(
    scope_summary_df,
    FIGURES_DIR / "anchor_context_hostility_rates.png",
    min_comments=MIN_CONTEXT_COMMENTS,
    top_n=15,
)
display(Image(filename=str(CONTEXT_FIGURE)))

## 10. Pares ofensa-entidad y plantillas lingüísticas

Estos resultados son los más cercanos a una orientación directa, pero suelen tener frecuencias pequeñas. Las plantillas sustituyen el nombre y la ofensa por `<TARGET>` y `<OFFENSE>` para facilitar la lectura agregada sin reproducir comentarios violentos completos.

In [ ]:
offense_target_pairs_df = build_offense_target_pairs(evidence_df)
relation_templates_df = build_relation_templates(prepared_df, evidence_df)

print("Pares ofensa-entidad de alta confianza:")
display(offense_target_pairs_df.head(40))
print("Plantillas objetivo-ofensa:")
display(relation_templates_df.head(40))

## 11. Entidades no catalogadas para revisión

La NER de spaCy se aplica únicamente a posts madre periodísticos vinculados a comentarios hostiles. El resultado es una lista local para revisión y no debe publicarse automáticamente ni incorporarse al catálogo sin comprobar que se trate de un actor público relevante.

In [ ]:
if ENABLE_NER_DISCOVERY and nlp is not None:
    uncatalogued_entities_df = discover_uncatalogued_source_entities(
        prepared_df,
        target_catalog_df,
        nlp,
        min_source_posts=NER_MIN_SOURCE_POSTS,
    )
    print("Entidades no catalogadas candidatas:", len(uncatalogued_entities_df))
else:
    uncatalogued_entities_df = pd.DataFrame()
    print("NER de descubrimiento omitida por configuración o falta de modelo.")

## 12. Muestra independiente para validar destinatarios

Esta muestra no modifica el etiquetado canónico de hostilidad y odio. Sus nuevas columnas permiten comprobar entidad, tipo de blanco, fundamento identitario y confianza de la inferencia.

In [ ]:
target_review_df = build_target_manual_review_candidate(
    prepared_df,
    assignments_df,
    sample_size=TARGET_REVIEW_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)
print("Filas preparadas para revisión manual de blancos:", len(target_review_df))
display(
    target_review_df[
        [
            "review_id",
            "event_id",
            "target_confidence",
            "primary_target_label",
            "primary_target_type",
            "hostility_pred",
            "hate_pred_experimental",
        ]
    ].head(20)
)

## 13. Exportación reproducible

La tabla de evidencia no incluye el texto. El texto solo permanece en el corpus enriquecido y en la muestra local destinada a revisión manual.

In [ ]:
target_catalog_df.to_csv(REPORTS_DIR / "target_catalog.csv", index=False)
entity_mention_hostility_df.to_csv(
    REPORTS_DIR / "entity_mention_hostility_selected.csv", index=False
)
mention_target_summary_df.to_csv(
    REPORTS_DIR / "political_entity_mentions_all_comments.csv", index=False
)
mention_type_summary_df.to_csv(
    REPORTS_DIR / "political_entity_mentions_by_type.csv", index=False
)
mention_overall_summary_df.to_csv(
    REPORTS_DIR / "political_entity_mentions_overall.csv", index=False
)
evidence_df.to_csv(REPORTS_DIR / "comment_target_evidence.csv", index=False)
assignments_df.to_csv(REPORTS_DIR / "comment_target_assignments.csv", index=False)
coverage_df.to_csv(REPORTS_DIR / "target_assignment_coverage.csv", index=False)
scope_summary_df.to_csv(REPORTS_DIR / "target_scope_summary.csv", index=False)
primary_target_summary_df.to_csv(REPORTS_DIR / "primary_target_summary.csv", index=False)
target_type_summary_df.to_csv(REPORTS_DIR / "target_type_summary.csv", index=False)
offense_target_pairs_df.to_csv(REPORTS_DIR / "offense_target_pairs.csv", index=False)
relation_templates_df.to_csv(REPORTS_DIR / "target_relation_templates.csv", index=False)
uncatalogued_entities_df.to_csv(
    REPORTS_DIR / "uncatalogued_source_entities_for_review.csv", index=False
)
target_review_df.to_csv(
    REPORTS_DIR / "target_manual_review_candidate.csv", index=False
)

assignment_columns = [
    "tweet_id",
    "primary_target_id",
    "primary_target_label",
    "primary_target_type",
    "primary_target_potential_identity_basis",
    "target_confidence",
    "target_assignment_basis",
    "target_candidate_ids",
    "target_candidate_labels",
    "direct_linked_target_count",
    "direct_target_count",
    "anchor_target_count",
    "all_catalog_target_count",
    "offense_present",
    "offense_families",
]
enriched_corpus_df = corpus_df.merge(
    assignments_df[assignment_columns],
    on="tweet_id",
    how="left",
    validate="one_to_one",
)
enriched_corpus_df["target_analysis_version"] = "0.1.0"
ENRICHED_CORPUS_PATH = (
    DATA_PROCESSED
    / "x_media_anchored_interactions_corpus_formal_targets_exploratory.csv"
)
enriched_corpus_df.to_csv(ENRICHED_CORPUS_PATH, index=False)

hostile_assignments = assignments_df.loc[assignments_df["hostility_pred"].eq(1)]
hate_assignments = assignments_df.loc[assignments_df["hate_pred_experimental"].eq(1)]
summary_df = pd.DataFrame(
    [
        {"metric": "corpus_comments", "value": len(assignments_df)},
        {"metric": "hostility_balanced_positive", "value": len(hostile_assignments)},
        {"metric": "hate_balanced_experimental_positive", "value": len(hate_assignments)},
        {
            "metric": "hostile_high_confidence_target",
            "value": int(hostile_assignments["target_confidence"].eq("high").sum()),
        },
        {
            "metric": "hostile_medium_confidence_target",
            "value": int(hostile_assignments["target_confidence"].eq("medium").sum()),
        },
        {
            "metric": "hostile_low_confidence_target",
            "value": int(hostile_assignments["target_confidence"].eq("low").sum()),
        },
        {
            "metric": "hostile_ambiguous_target",
            "value": int(hostile_assignments["target_confidence"].eq("ambiguous").sum()),
        },
        {
            "metric": "hate_high_confidence_target_experimental",
            "value": int(hate_assignments["target_confidence"].eq("high").sum()),
        },
        {"metric": "catalog_targets", "value": len(target_catalog_df)},
        {"metric": "validated_offense_families", "value": len(offense_patterns_df)},
    ]
)
summary_df.to_csv(REPORTS_DIR / "target_analysis_summary.csv", index=False)

display(summary_df)
print("Corpus enriquecido:", ENRICHED_CORPUS_PATH)
print("Muestra manual independiente:", REPORTS_DIR / "target_manual_review_candidate.csv")

## 14. Advertencias metodológicas

- Una entidad mencionada no es necesariamente el blanco del ataque.
- El contexto del post madre indica el tema de conversación, no el destinatario confirmado del comentario.
- Los conteos directos pueden incluir comentarios que mencionan varias entidades; por eso no siempre son mutuamente excluyentes.
- Las comparaciones entre actores deben normalizarse por exposición y acompañarse del tamaño de muestra.
- Atacar a una persona política no equivale automáticamente a discurso de odio. Para odio debe validarse que el ataque se fundamente en identidad, cultura o adscripción ideológica.
- `potential_identity_basis` solo señala una dimensión posible; nunca completa automáticamente `manual_hate_speech`.
- El modelo de odio continúa siendo experimental y tiene pocos positivos manuales.
- Las asignaciones de confianza alta también deben auditarse: proximidad y sintaxis reducen falsos positivos, pero no resuelven ironía, citas, negación o correferencia compleja.
- La lista de entidades descubiertas por NER es únicamente una cola de revisión local.

El resultado apropiado para el TFM es presentar por separado evidencia directa, evidencia contextual y casos validados manualmente.